# 02 - ROGII Baseline Modeling

This notebook combines the simple deterministic baselines and the feature-baseline tree model for the ROGII Wellbore Geology Prediction competition.

Workflow:

1. Load train/test wells and the submission template.
2. Build inference-safe rolling features from `GR`, `TVT_input`, `MD`, and coordinates.
3. Evaluate deterministic baselines: carry-forward, linear trend, damped trend, and blends.
4. Train a residual tree model using held-out-well masked-tail validation.
5. Generate `submission.csv`, using the feature model only if validation beats carry-forward.

The notebook does not use train-only geology-top columns such as `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, or `BUDA`.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)

RANDOM_STATE = 42
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'
DATA_ROOT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / 'competitions' / COMPETITION_SLUG,
    KAGGLE_INPUT_ROOT / COMPETITION_SLUG,
]
WORK_DIR = Path('/kaggle/working')
SUBMISSION_PATH = WORK_DIR / 'submission.csv'


def resolve_data_root(candidates):
    for candidate in candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate
    for sample_file in KAGGLE_INPUT_ROOT.rglob('sample_submission.csv') if KAGGLE_INPUT_ROOT.exists() else []:
        if COMPETITION_SLUG in sample_file.as_posix():
            return sample_file.parent
    return candidates[0]


DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
print('DATA_ROOT:', DATA_ROOT)
print('sample_submission exists:', (DATA_ROOT / 'sample_submission.csv').exists())

DATA_ROOT: /kaggle/input/competitions/rogii-wellbore-geology-prediction
sample_submission exists: True


## 1. Load Files

The notebook uses only horizontal well CSVs and `sample_submission.csv`. Typewell alignment is deliberately left for the next modeling stage so we can first measure whether basic rolling features improve on carry-forward.

In [2]:
def find_files(root: Path, pattern: str):
    return sorted(root.rglob(pattern)) if root.exists() else []


def well_name_from_horizontal_path(path: Path) -> str:
    return path.name.split('__horizontal_well.csv')[0]


def parse_submission_id(value):
    well, row = str(value).rsplit('_', 1)
    return well, int(row)


def get_column(df: pd.DataFrame, name: str):
    lookup = {col.lower(): col for col in df.columns}
    return lookup.get(name.lower())


train_files = find_files(DATA_ROOT / 'train', '*__horizontal_well.csv')
test_files = find_files(DATA_ROOT / 'test', '*__horizontal_well.csv')
sample_submission = pd.read_csv(DATA_ROOT / 'sample_submission.csv')

id_col = sample_submission.columns[0]
target_col = 'tvt' if 'tvt' in sample_submission.columns else sample_submission.columns[-1]
parsed_ids = sample_submission[id_col].map(parse_submission_id)
sample_submission['well'] = [item[0] for item in parsed_ids]
sample_submission['row_idx'] = [item[1] for item in parsed_ids]

print('train wells:', len(train_files))
print('test wells:', len(test_files))
print('submission rows:', len(sample_submission))
display(sample_submission.head())

train wells: 773
test wells: 3
submission rows: 14151


,id,tvt,well,row_idx
0,000d7d20_1442,0.0,000d7d20,1442
1,000d7d20_1443,0.0,000d7d20,1443
2,000d7d20_1444,0.0,000d7d20,1444
3,000d7d20_1445,0.0,000d7d20,1445
4,000d7d20_1446,0.0,000d7d20,1446


## 2. Feature Engineering

Features are designed to be available at inference time. The model never uses train-only geology tops such as `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, or `BUDA`.

Feature families:

- relative position and measured-depth features;
- per-well centered/normalized coordinates;
- `GR` missingness, interpolation, and rolling statistics;
- carry-forward `TVT_input` features;
- distance from the last known `TVT_input` point;
- recent `TVT_input` slope and local volatility.

The tree model predicts residuals over carry-forward rather than absolute TVT. This makes the baseline safer: if the model cannot improve, the fallback is still carry-forward.

Validation-only columns such as `tail_fraction` are excluded from model features so the same feature list is available when predicting real test wells.

In [3]:
ROLL_WINDOWS = (25, 101, 301)
FEATURE_COLUMNS = None
NON_FEATURE_COLUMNS = {'well', 'target_tvt', 'target_residual', 'tail_fraction'}


def numeric_col(df, name, default=np.nan):
    col = get_column(df, name)
    if col is None:
        return pd.Series(default, index=df.index, dtype='float64')
    return pd.to_numeric(df[col], errors='coerce').astype('float64')


def tvt_input_series(df):
    tvt_input_col = get_column(df, 'TVT_input')
    tvt_col = get_column(df, 'TVT')
    if tvt_input_col is not None:
        return pd.to_numeric(df[tvt_input_col], errors='coerce').astype('float64').reset_index(drop=True)
    if tvt_col is not None:
        return pd.to_numeric(df[tvt_col], errors='coerce').astype('float64').reset_index(drop=True)
    return pd.Series(np.nan, index=range(len(df)), dtype='float64')


def carry_forward_prediction(df):
    y_input = tvt_input_series(df)
    carry = y_input.ffill().bfill()
    if carry.isna().all():
        carry = pd.Series(0.0, index=range(len(df)), dtype='float64')
    return carry.astype('float64')


def add_rolling_features(out, source, prefix):
    for window in ROLL_WINDOWS:
        rolled = source.rolling(window=window, min_periods=1)
        out[f'{prefix}_roll_mean_{window}'] = rolled.mean()
        out[f'{prefix}_roll_std_{window}'] = rolled.std().fillna(0.0)
        out[f'{prefix}_roll_min_{window}'] = rolled.min()
        out[f'{prefix}_roll_max_{window}'] = rolled.max()
    return out


def build_features(df, well):
    n = len(df)
    idx = pd.Series(np.arange(n), dtype='float64')
    denom = max(n - 1, 1)

    md = numeric_col(df, 'MD').reset_index(drop=True)
    x = numeric_col(df, 'X').reset_index(drop=True)
    y = numeric_col(df, 'Y').reset_index(drop=True)
    z = numeric_col(df, 'Z').reset_index(drop=True)
    gr_raw = numeric_col(df, 'GR').reset_index(drop=True)
    y_input = tvt_input_series(df)
    carry = carry_forward_prediction(df)

    gr_interp = gr_raw.interpolate(limit_direction='both').ffill().bfill()
    if gr_interp.isna().all():
        gr_interp = pd.Series(0.0, index=range(n), dtype='float64')

    known = y_input.notna()
    known_idx = pd.Series(np.where(known, idx, np.nan)).ffill().fillna(0.0)
    distance_from_known = (idx - known_idx).clip(lower=0)
    hidden_flag = (~known).astype('int8')

    tvt_diff = carry.diff().fillna(0.0)
    recent_slope = tvt_diff.rolling(window=101, min_periods=1).mean().fillna(0.0)
    recent_volatility = tvt_diff.rolling(window=101, min_periods=1).std().fillna(0.0)

    out = pd.DataFrame({
        'well': well,
        'row_idx': idx,
        'n_rows': float(n),
        'rel_pos': idx / denom,
        'distance_from_known': distance_from_known,
        'distance_from_known_frac': distance_from_known / denom,
        'hidden_flag': hidden_flag,
        'md': md,
        'md_rel': (md - md.min()) / (md.max() - md.min()) if md.notna().sum() > 1 and md.max() != md.min() else idx / denom,
        'x_centered': x - x.mean(),
        'y_centered': y - y.mean(),
        'z_centered': z - z.mean(),
        'gr': gr_raw,
        'gr_interp': gr_interp,
        'gr_missing': gr_raw.isna().astype('int8'),
        'gr_centered': gr_interp - gr_interp.mean(),
        'carry_tvt': carry,
        'tvt_recent_slope': recent_slope,
        'tvt_recent_volatility': recent_volatility,
    })

    out = add_rolling_features(out, gr_interp, 'gr')
    out = add_rolling_features(out, carry, 'carry_tvt')

    numeric_features = [col for col in out.columns if col != 'well']
    out[numeric_features] = out[numeric_features].replace([np.inf, -np.inf], np.nan)
    out[numeric_features] = out[numeric_features].fillna(out[numeric_features].median(numeric_only=True)).fillna(0.0)
    return out


def make_masked_frame(df, tail_fraction):
    tvt_col = get_column(df, 'TVT')
    if tvt_col is None:
        return None, None, None
    y_true = numeric_col(df, 'TVT').reset_index(drop=True)
    eval_start = int(len(df) * (1 - tail_fraction))
    masked = df.copy().reset_index(drop=True)
    masked['TVT_input'] = y_true.copy()
    masked.loc[eval_start:, 'TVT_input'] = np.nan
    return masked, y_true, eval_start


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))) if mask.any() else np.nan


def select_feature_columns(frame):
    return [col for col in frame.columns if col not in NON_FEATURE_COLUMNS]


def align_feature_frame(frame):
    aligned = frame.reindex(columns=FEATURE_COLUMNS, fill_value=0.0).copy()
    aligned = aligned.replace([np.inf, -np.inf], np.nan)
    return aligned.fillna(0.0)


# quick smoke test on one train well
if train_files:
    sample_df = pd.read_csv(train_files[0])
    masked_df, y_true, eval_start = make_masked_frame(sample_df, tail_fraction=0.30)
    sample_features = build_features(masked_df, well_name_from_horizontal_path(train_files[0]))
    print('sample feature shape:', sample_features.shape)
    display(sample_features.head())

sample feature shape: (5278, 43)


,well,row_idx,n_rows,rel_pos,distance_from_known,distance_from_known_frac,hidden_flag,md,md_rel,x_centered,y_centered,z_centered,gr,gr_interp,gr_missing,gr_centered,carry_tvt,tvt_recent_slope,tvt_recent_volatility,gr_roll_mean_25,gr_roll_std_25,gr_roll_min_25,gr_roll_max_25,gr_roll_mean_101,gr_roll_std_101,gr_roll_min_101,gr_roll_max_101,gr_roll_mean_301,gr_roll_std_301,gr_roll_min_301,gr_roll_max_301,carry_tvt_roll_mean_25,carry_tvt_roll_std_25,carry_tvt_roll_min_25,carry_tvt_roll_max_25,carry_tvt_roll_mean_101,carry_tvt_roll_std_101,carry_tvt_roll_min_101,carry_tvt_roll_max_101,carry_tvt_roll_mean_301,carry_tvt_roll_std_301,carry_tvt_roll_min_301,carry_tvt_roll_max_301
0,000d7d20,0.0,5278.0,0.000000,0.0,0.0,0,11467.0,0.000000,11.278414,-2395.583429,414.624324,115.692586,115.692586,0,20.475226,11236.02,0.000,0.000000,115.692586,0.000000,115.692586,115.692586,115.692586,0.000000,115.692586,115.692586,115.692586,0.000000,115.692586,115.692586,11236.020000,0.000000,11236.02,11236.02,11236.020000,0.000000,11236.02,11236.02,11236.020000,0.000000,11236.02,11236.02
1,000d7d20,1.0,5278.0,0.000190,0.0,0.0,0,11468.0,0.000190,11.298414,-2395.373429,413.644324,115.584293,115.584293,0,20.366933,11237.05,0.515,0.728320,115.638440,0.076574,115.584293,115.692586,115.638440,0.076574,115.584293,115.692586,115.638440,0.076574,115.584293,115.692586,11236.535000,0.728320,11236.02,11237.05,11236.535000,0.728320,11236.02,11237.05,11236.535000,0.728320,11236.02,11237.05
2,000d7d20,2.0,5278.0,0.000379,0.0,0.0,0,11469.0,0.000379,11.318414,-2395.153429,412.674324,135.446960,135.446960,0,40.229599,11238.09,0.690,0.597578,122.241280,11.436583,115.584293,135.446960,122.241280,11.436583,115.584293,135.446960,122.241280,11.436583,115.584293,135.446960,11237.053333,1.035004,11236.02,11238.09,11237.053333,1.035004,11236.02,11238.09,11237.053333,1.035004,11236.02,11238.09
3,000d7d20,3.0,5278.0,0.000569,0.0,0.0,0,11470.0,0.000569,11.338414,-2394.943429,411.694324,140.401346,140.401346,0,45.183986,11239.12,0.775,0.516688,126.781296,13.024744,115.584293,140.401346,126.781296,13.024744,115.584293,140.401346,126.781296,13.024744,115.584293,140.401346,11237.570000,1.334891,11236.02,11239.12,11237.570000,1.334891,11236.02,11239.12,11237.570000,1.334891,11236.02,11239.12
4,000d7d20,4.0,5278.0,0.000758,0.0,0.0,0,11471.0,0.000758,11.368414,-2394.723429,410.724324,111.270638,111.270638,0,16.053278,11240.15,0.826,0.461768,123.679165,13.241943,111.270638,140.401346,123.679165,13.241943,111.270638,140.401346,123.679165,13.241943,111.270638,140.401346,11238.086000,1.633319,11236.02,11240.15,11238.086000,1.633319,11236.02,11240.15,11238.086000,1.633319,11236.02,11240.15


## 3. Build Train And Validation Samples

Validation is by held-out wells, not random rows. This is stricter because it asks whether the model generalizes to wells it did not see during training.

To keep runtime reasonable, each simulated hidden suffix is downsampled to a fixed number of rows. The model still sees many wells and multiple hidden-window lengths.

In [4]:
TAIL_FRACTIONS = (0.20, 0.30, 0.40)
MAX_TRAIN_WELLS = 520
MAX_VALIDATION_WELLS = 160
MAX_ROWS_PER_WELL_FOLD = 900
VALIDATION_WELL_FRACTION = 0.20


def sample_eval_rows(features, y_true, eval_start, max_rows, random_state):
    eval_idx = np.arange(eval_start, len(features))
    if len(eval_idx) > max_rows:
        rng = np.random.default_rng(random_state)
        eval_idx = np.sort(rng.choice(eval_idx, size=max_rows, replace=False))
    sampled = features.iloc[eval_idx].copy()
    sampled['target_tvt'] = y_true.iloc[eval_idx].to_numpy()
    sampled['target_residual'] = sampled['target_tvt'] - sampled['carry_tvt']
    return sampled


def build_modeling_table(files, max_wells, tail_fractions, max_rows_per_fold, seed=RANDOM_STATE):
    frames = []
    for well_number, path in enumerate(files[:max_wells]):
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        for frac_number, tail_fraction in enumerate(tail_fractions):
            masked, y_true, eval_start = make_masked_frame(df, tail_fraction)
            if masked is None:
                continue
            features = build_features(masked, well)
            sampled = sample_eval_rows(
                features,
                y_true,
                eval_start,
                max_rows=max_rows_per_fold,
                random_state=seed + 1000 * well_number + frac_number,
            )
            sampled['tail_fraction'] = tail_fraction
            frames.append(sampled)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


rng = np.random.default_rng(RANDOM_STATE)
train_paths = np.array(train_files[:MAX_TRAIN_WELLS], dtype=object)
rng.shuffle(train_paths)
valid_size = max(1, int(len(train_paths) * VALIDATION_WELL_FRACTION))
valid_paths = list(train_paths[:valid_size])
model_train_paths = list(train_paths[valid_size:])

train_table = build_modeling_table(model_train_paths, MAX_TRAIN_WELLS, TAIL_FRACTIONS, MAX_ROWS_PER_WELL_FOLD)
valid_table = build_modeling_table(valid_paths, MAX_VALIDATION_WELLS, TAIL_FRACTIONS, MAX_ROWS_PER_WELL_FOLD)

FEATURE_COLUMNS = select_feature_columns(train_table)

print('model train wells:', len(model_train_paths))
print('validation wells:', len(valid_paths))
print('train table:', train_table.shape)
print('valid table:', valid_table.shape)
print('feature count:', len(FEATURE_COLUMNS))
display(train_table.head())

model train wells: 416
validation wells: 104
train table: (1121693, 46)
valid table: (280156, 46)
feature count: 42


,well,row_idx,n_rows,rel_pos,distance_from_known,distance_from_known_frac,hidden_flag,md,md_rel,x_centered,y_centered,z_centered,gr,gr_interp,gr_missing,gr_centered,carry_tvt,tvt_recent_slope,tvt_recent_volatility,gr_roll_mean_25,gr_roll_std_25,gr_roll_min_25,gr_roll_max_25,gr_roll_mean_101,gr_roll_std_101,gr_roll_min_101,gr_roll_max_101,gr_roll_mean_301,gr_roll_std_301,gr_roll_min_301,gr_roll_max_301,carry_tvt_roll_mean_25,carry_tvt_roll_std_25,carry_tvt_roll_min_25,carry_tvt_roll_max_25,carry_tvt_roll_mean_101,carry_tvt_roll_std_101,carry_tvt_roll_min_101,carry_tvt_roll_max_101,carry_tvt_roll_mean_301,carry_tvt_roll_std_301,carry_tvt_roll_min_301,carry_tvt_roll_max_301,target_tvt,target_residual,tail_fraction
0,89f36adf,5636.0,7041.0,0.800568,5.0,0.000710,1,16974.0,0.800568,-52.69703,-2090.112683,-94.874107,83.761381,73.195731,1,-10.852761,12229.54,0.004455,0.007547,83.950319,7.505366,67.090881,98.762275,85.933816,6.824839,67.090881,100.431881,88.708312,9.877856,62.073547,112.093564,12229.5680,0.024665,12229.54,12229.61,12229.494356,0.147692,12229.10,12229.63,12227.828306,1.537463,12225.57,12229.63,12229.53,-0.01,0.2
1,89f36adf,5638.0,7041.0,0.800852,7.0,0.000994,1,16976.0,0.800852,-52.75703,-2092.112683,-94.924107,83.761381,71.252118,1,-12.796375,12229.54,0.004158,0.007385,83.050606,8.766220,63.751670,98.762275,85.507947,7.299456,63.751670,100.431881,88.524112,10.015577,62.073547,112.093564,12229.5624,0.022226,12229.54,12229.60,12229.502871,0.137203,12229.13,12229.63,12227.854153,1.533304,12225.57,12229.63,12229.52,-0.02,0.2
2,89f36adf,5639.0,7041.0,0.800994,8.0,0.001136,1,16977.0,0.800994,-52.78703,-2093.102683,-94.944107,78.752565,78.752565,0,-5.295928,12229.54,0.004059,0.007373,83.184515,8.669597,63.751670,98.762275,85.408847,7.322747,63.751670,100.431881,88.446541,10.000477,62.073547,112.093564,12229.5600,0.021213,12229.54,12229.60,12229.506931,0.132028,12229.15,12229.63,12227.867110,1.531008,12225.57,12229.63,12229.52,-0.02,0.2
3,89f36adf,5640.0,7041.0,0.801136,9.0,0.001278,1,16978.0,0.801136,-52.81703,-2094.102683,-94.964107,83.761381,80.831053,1,-3.217439,12229.54,0.003861,0.007207,83.368172,8.564340,63.751670,98.762275,85.305614,7.312958,63.751670,100.431881,88.398077,10.001922,62.073547,112.093564,12229.5576,0.019850,12229.54,12229.60,12229.510792,0.127096,12229.16,12229.63,12227.880100,1.528549,12225.57,12229.63,12229.52,-0.02,0.2
4,89f36adf,5641.0,7041.0,0.801278,10.0,0.001420,1,16979.0,0.801278,-52.85703,-2095.102683,-94.994107,83.761381,82.909542,1,-1.138951,12229.54,0.003762,0.007190,83.601576,8.464602,63.751670,98.762275,85.198249,7.267149,63.751670,100.431881,88.378719,10.006903,62.073547,112.093564,12229.5552,0.018056,12229.54,12229.59,12229.514554,0.122135,12229.18,12229.63,12227.893123,1.525926,12225.57,12229.63,12229.51,-0.03,0.2


## 4. Deterministic Baseline Comparison

Before training a tree model, compare inference-safe deterministic baselines on the same held-out validation wells. This keeps the modeling decision grounded: a feature model has to beat carry-forward, not just produce a submission.

In [5]:
def linear_trend_prediction(df, tail_points=500):
    x = numeric_col(df, 'MD').reset_index(drop=True)
    if x.notna().sum() < 2:
        x = pd.Series(np.arange(len(df), dtype='float64'))
    y_input = tvt_input_series(df)
    carry = carry_forward_prediction(df)
    known = y_input.notna() & x.notna()
    if known.sum() < 2:
        return carry

    x_known = x[known].to_numpy()
    y_known = y_input[known].to_numpy()
    n_tail = min(tail_points, len(x_known))
    x_tail = x_known[-n_tail:]
    y_tail = y_known[-n_tail:]
    x_anchor = x_tail[-1]
    y_anchor = y_tail[-1]
    if np.nanstd(x_tail) == 0:
        return carry

    slope = np.polyfit(x_tail - x_anchor, y_tail - y_anchor, 1)[0]
    pred = y_anchor + slope * (x.to_numpy() - x_anchor)
    pred = pd.Series(pred, index=range(len(df)), dtype='float64')
    pred[y_input.notna()] = y_input[y_input.notna()]
    return pred.ffill().bfill().astype('float64')


def damped_trend_prediction(df, damp=0.35):
    carry = carry_forward_prediction(df)
    trend = linear_trend_prediction(df)
    y_input = tvt_input_series(df)
    pred = carry + damp * (trend - carry)
    pred[y_input.notna()] = y_input[y_input.notna()]
    return pred.astype('float64')


def blended_prediction(df, weight=0.25):
    carry = carry_forward_prediction(df)
    trend = linear_trend_prediction(df)
    y_input = tvt_input_series(df)
    pred = (1 - weight) * carry + weight * trend
    pred[y_input.notna()] = y_input[y_input.notna()]
    return pred.astype('float64')


def deterministic_prediction(df, model_name):
    if model_name == 'carry_forward':
        return carry_forward_prediction(df)
    if model_name == 'linear_trend':
        return linear_trend_prediction(df)
    if model_name == 'damped_trend_035':
        return damped_trend_prediction(df, damp=0.35)
    if model_name == 'blend_025':
        return blended_prediction(df, weight=0.25)
    if model_name == 'blend_050':
        return blended_prediction(df, weight=0.50)
    raise ValueError(model_name)


def evaluate_deterministic_baselines(paths, tail_fractions):
    model_names = ['carry_forward', 'blend_025', 'damped_trend_035', 'blend_050', 'linear_trend']
    rows = []
    for path in paths:
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        for tail_fraction in tail_fractions:
            masked, y_true, eval_start = make_masked_frame(df, tail_fraction)
            if masked is None:
                continue
            for model_name in model_names:
                pred = deterministic_prediction(masked, model_name)
                rows.append({
                    'well': well,
                    'tail_fraction': tail_fraction,
                    'model': model_name,
                    'rmse': rmse(y_true.iloc[eval_start:], pred.iloc[eval_start:]),
                })
    return pd.DataFrame(rows)


simple_validation = evaluate_deterministic_baselines(valid_paths, TAIL_FRACTIONS)
simple_summary = (
    simple_validation.groupby('model')['rmse']
    .agg(['mean', 'median', 'std', 'count'])
    .sort_values('mean')
)
display(simple_summary)

deterministic_winner = simple_summary.index[0]
print('best deterministic baseline:', deterministic_winner)

,mean,median,std,count
model,,,,
carry_forward,8.145602,6.424366,6.274099,312
blend_025,9.117095,7.120719,6.752397,312
damped_trend_035,9.802191,7.678469,7.429644,312
blend_050,11.016623,8.697231,8.836815,312
linear_trend,15.961054,11.404072,15.146480,312


best deterministic baseline: carry_forward


## 5. Train Feature Baseline

The feature model predicts residuals over carry-forward. Validation reports both the raw carry-forward score and the feature-tree score.

In [6]:
def fit_feature_model(train_table):
    X = align_feature_frame(train_table)
    y = train_table['target_residual']
    try:
        model = HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=31,
            min_samples_leaf=40,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        )
        model.fit(X, y)
        model_name = 'HistGradientBoostingRegressor'
    except Exception as exc:
        print('HistGradientBoostingRegressor failed; falling back to RandomForestRegressor:', repr(exc))
        model = RandomForestRegressor(
            n_estimators=160,
            max_depth=10,
            min_samples_leaf=20,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        model.fit(X, y)
        model_name = 'RandomForestRegressor'
    return model, model_name


model, model_name = fit_feature_model(train_table)
print('model:', model_name)

valid_pred_residual = model.predict(align_feature_frame(valid_table))
valid_tree_pred = valid_table['carry_tvt'].to_numpy() + valid_pred_residual
carry_rmse = rmse(valid_table['target_tvt'], valid_table['carry_tvt'])
tree_rmse = rmse(valid_table['target_tvt'], valid_tree_pred)

print('carry_forward validation RMSE:', carry_rmse)
print('feature_tree validation RMSE:', tree_rmse)
print('delta RMSE:', tree_rmse - carry_rmse)

valid_scored = valid_table[['well', 'tail_fraction', 'target_tvt', 'carry_tvt']].copy()
valid_scored['feature_tree'] = valid_tree_pred
valid_summary = (
    valid_scored
    .groupby(['tail_fraction'])
    .apply(lambda x: pd.Series({
        'carry_rmse': rmse(x['target_tvt'], x['carry_tvt']),
        'feature_tree_rmse': rmse(x['target_tvt'], x['feature_tree']),
        'rows': len(x),
        'wells': x['well'].nunique(),
    }))
    .reset_index()
)
valid_summary['delta'] = valid_summary['feature_tree_rmse'] - valid_summary['carry_rmse']
display(valid_summary)

use_tree_model = bool(tree_rmse < carry_rmse)
print('use_tree_model_for_submission:', use_tree_model)

model: HistGradientBoostingRegressor
carry_forward validation RMSE: 10.280696473884893
feature_tree validation RMSE: 10.083950915468908
delta RMSE: -0.1967455584159854


,tail_fraction,carry_rmse,feature_tree_rmse,rows,wells,delta
0,0.2,8.545366,8.338058,92956.0,104.0,-0.207307
1,0.3,9.421048,9.170969,93600.0,104.0,-0.250079
2,0.4,12.452869,12.296626,93600.0,104.0,-0.156243


use_tree_model_for_submission: True


## 6. Refit And Generate Submission

If validation improves, refit the feature model on simulated tails from more training wells and use it for the test hidden rows. If validation does not improve, fall back to carry-forward. This makes the notebook safe to submit while still testing the feature baseline.

In [7]:
if use_tree_model:
    refit_table = build_modeling_table(train_files, max_wells=min(len(train_files), 720), tail_fractions=TAIL_FRACTIONS, max_rows_per_fold=MAX_ROWS_PER_WELL_FOLD)
    FEATURE_COLUMNS = select_feature_columns(refit_table)
    model, model_name = fit_feature_model(refit_table)
    print('refit model:', model_name)
    print('refit table:', refit_table.shape)
else:
    print('Validation did not beat carry-forward. Submission will use carry-forward fallback.')


def predict_test_well(df, well):
    features = build_features(df.reset_index(drop=True), well)
    carry = features['carry_tvt'].to_numpy()
    if use_tree_model:
        residual = model.predict(align_feature_frame(features))
        pred = carry + residual
        known = tvt_input_series(df).notna().to_numpy()
        pred[known] = tvt_input_series(df).to_numpy()[known]
        return pd.Series(pred, index=range(len(df)), dtype='float64')
    return pd.Series(carry, index=range(len(df)), dtype='float64')


test_lookup = {well_name_from_horizontal_path(path): path for path in test_files}
well_predictions = {}
for well, path in test_lookup.items():
    df = pd.read_csv(path)
    well_predictions[well] = predict_test_well(df, well).reset_index(drop=True)

fallback = 0.0
non_empty = [pred.dropna().to_numpy() for pred in well_predictions.values() if pred.dropna().size]
if non_empty:
    fallback = float(np.nanmedian(np.concatenate(non_empty)))

submission = sample_submission[[id_col]].copy()
values = []
missing_wells = set()
for sub_id in sample_submission[id_col]:
    well, row_idx = parse_submission_id(sub_id)
    pred = well_predictions.get(well)
    if pred is None or len(pred) == 0:
        missing_wells.add(well)
        values.append(fallback)
    elif 0 <= row_idx < len(pred):
        values.append(float(pred.iloc[row_idx]))
    else:
        values.append(float(pred.iloc[-1]))

submission[target_col] = values
submission.to_csv(SUBMISSION_PATH, index=False)

print('wrote:', SUBMISSION_PATH)
print('selected submission model:', 'feature_tree' if use_tree_model else 'carry_forward')
print('rows:', len(submission))
print('missing wells:', len(missing_wells))
display(submission.head())
display(submission[target_col].describe())

refit model: HistGradientBoostingRegressor
refit table: (1941042, 46)
wrote: /kaggle/working/submission.csv
selected submission model: feature_tree
rows: 14151
missing wells: 0


,id,tvt
0,000d7d20_1442,11741.930549
1,000d7d20_1443,11741.930549
2,000d7d20_1444,11741.930549
3,000d7d20_1445,11741.930549
4,000d7d20_1446,11742.087811


count    14151.000000
mean     11904.612509
std        276.136758
min      11602.830165
25%      11606.543223
50%      11747.341908
75%      12221.787200
max      12229.986829
Name: tvt, dtype: float64

## 7. Readout

Use the validation delta as the submission decision:

- if `feature_tree_rmse < carry_rmse`, the feature baseline earned a submission;
- if not, the notebook intentionally falls back to carry-forward.

Current public-score readout:

- V3 public score: `15.883`;
- V6 public score: `15.491`;
- V7 public score: `15.491`.

The feature tree improved over the older baseline level, but the latest rerun matched the current best instead of improving it. Treat this as a plateau for rolling `GR`/`TVT_input` features. The next experiment should add typewell-alignment features, because that is a new source of geological signal rather than another tuning pass over the same feature family.
